# 0. Hướng dẫn sử dụng và chống test leakage
Notebook điều phối Lab Day 2. Chưa có kết quả thực nghiệm trong bản Git này.
Chạy từng phần, đọc điều kiện rồi chủ động bật cờ tương ứng. Mặc định mọi thao tác cài đặt, mạng, dữ liệu, training, inference và ghi báo cáo đều tắt.
Train chỉ cập nhật trọng số; validation chọn checkpoint/backbone/recipe/phương pháp và fit temperature. Test chỉ mở sau khi khóa cấu hình, mỗi seed một lượt qua loader (TTA có K forward trong cùng lượt).
Không thay cấu hình sau khi mở test. Không dùng test để cập nhật BN. Fold bắt buộc là 0. Không gộp val vào train.
Bản notebook chạy thật có thể **Save a copy in Drive**. Bản Git phải giữ outputs rỗng.
Nhận xét sau khi chạy: **[Điền bằng artifact và số đo thật; chưa có kết luận.]**


# 1. Thiết lập môi trường Colab/Kaggle
Colab: GPU không được đảm bảo. Kaggle: bật Internet khi clone/cài gói; Attach Dataset trước khi chuẩn bị dữ liệu.
Cờ cài gói ở phần 3: đổi thành `INSTALL_DEPS=True` khi chủ động cài, rồi chạy cell cài đặt ở phần 4. Không cài lại torch/torchvision.


In [ ]:
import sys
import os
import json
import subprocess
import hashlib
import shutil
import zipfile
import gc
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from dataclasses import asdict, replace
from contextlib import nullcontext

IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
IN_KAGGLE = Path("/kaggle/working").is_dir()
PLATFORM = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"
def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"
print("Platform:", PLATFORM, "Python:", sys.version)
print("torch:", package_version("torch"), "torchvision:", package_version("torchvision"))
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except ImportError:
    print("torch chưa có; hãy dùng runtime Colab/Kaggle đã cung cấp torch.")


# 2. Clone/cập nhật repo và import code
Bật RUN_REPO_SETUP tại phần 3 rồi chạy lại cell dưới. Nếu đang mở notebook trong repo đã clone, tự tìm root.
Repo có thay đổi chưa lưu sẽ bị từ chối pull. Pull dùng fast-forward; không reset hoặc tự push.


In [ ]:
REPO_URL = "https://github.com/naoh-pt/K4-Track4-Day2-PhanTrongHoan-2A202602954.git"
REPO_NAME = "K4-Track4-Day2-PhanTrongHoan-2A202602954"
candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / "eval.py").is_file() and (p / "code").is_dir()),
                 Path("/kaggle/working" if IN_KAGGLE else "/content") / REPO_NAME)

def setup_repo():
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        dirty = subprocess.check_output(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True)
        if dirty.strip():
            raise RuntimeError("Repo có thay đổi: lưu bản notebook chạy ở ngoài repo trước khi pull.")
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", "main"], check=True)
    print("Repo:", REPO_ROOT)

if globals().get("RUN_REPO_SETUP", False):
    setup_repo()


# 3. Cấu hình đường dẫn dữ liệu và output
**Cell cấu hình duy nhất:** sửa platform, đường dẫn và các lựa chọn tại đây.
Các đường dẫn /content và /kaggle là quy ước runtime; `<dien-dataset-slug>` phải thay bằng slug Dataset đã attach.
Colab lưu artifact trực tiếp trong Drive, ảnh train được copy/giải nén vào /content. Kaggle đọc /kaggle/input và ghi /kaggle/working.


In [ ]:
PLATFORM_MODE = "kaggle" if IN_KAGGLE else "colab"
DRIVE_DATA_DIR = "/content/drive/MyDrive/DeepWeeds"
KAGGLE_DATA_DIR = "/kaggle/input/<dien-dataset-slug>"
LOCAL_DATA_DIR = "/content/deepweeds_data" if PLATFORM_MODE == "colab" else "/kaggle/working/deepweeds_data"
OUTPUT_DIR = Path("/content/drive/MyDrive/DeepWeedsLabOutputs" if PLATFORM_MODE == "colab"
                  else "/kaggle/working/deepweeds_outputs")
EXPECTED_MD5 = "b7b30f96d466fba86016aa5a26606e0f"
EXPECTED_IMAGES = 17509

INSTALL_DEPS = False
USE_GOOGLE_DRIVE = False
RUN_REPO_SETUP = False
RUN_PREPARE_DATA = False
RUN_EDA = False
RUN_SANITY_CHECKS = False
RUN_BASELINE = False
RUN_BACKBONES = False
RUN_TRAINING_ABLATIONS = False
RUN_INFERENCE_EXPERIMENTS = False
RUN_LATENCY = False
RUN_FINAL_TRAINING = False
RUN_FINAL_TEST = False
RUN_EVAL_REPORTS = False
RUN_EXPORT_RESULTS = False
RUN_ARTIFACT_CHECK = False

SELECTED_BACKBONE = None
COMBINED_RECIPE = None  # Dict các thay đổi đã thắng trên val; không điền trước khi đo.
INFERENCE_CHECKPOINT = None  # Đường dẫn best.pt của run đã chọn bằng val.
ENSEMBLE_CHECKPOINTS = []  # Tùy chọn I05; cùng thứ tự ảnh và cùng tập val.
FINAL_BACKBONE = None
FINAL_TRAINING_CONFIG = None  # Dict override Config, chỉ các field recipe.
FINAL_INFERENCE_METHOD = None  # ID trong INFERENCE_SPECS; ví dụ chọn sau khi có bảng val.
FINAL_ENSEMBLE_CHECKPOINTS = {}  # Dict seed -> list checkpoint paths nếu final chọn I05.
FINAL_SEEDS = [0, 1, 2]
CONFIRM_FINAL_TEST = "NO"
CONFIRM_NO_TEST_PREDICTIONS = False
ALLOW_CPU_TRAINING = False
BASELINE_BATCH_SIZE = 64
INFERENCE_BATCH_SIZE = 32
LATENCY_WARMUP = 10
LATENCY_ITERATIONS = 50
LATENCY_K_VIEWS = None  # Chỉ đặt K sau khi chọn TTA; benchmark K forward cùng resolution.
LATENCY_P95_MS = None  # Chỉ điền p95 batch 1 CUDA đã đo cho đúng cấu hình final.
EVAL_UNCAL_PATTERN = None  # Có thể dùng final/predictions/F01uncal_seed*_test.csv
EVAL_FINAL_VAL_PATTERN = None  # Có thể dùng final/selected_validation/F01_seed*/F01_seed*_val.csv
CONFIRM_OVERWRITE_RESULTS = False

if PLATFORM_MODE not in {"colab", "kaggle"}:
    raise ValueError("PLATFORM_MODE phải là colab hoặc kaggle")


# 4. Chuẩn bị dữ liệu từ Google Drive hoặc Kaggle Dataset
Colab: upload images.zip và thư mục labels/ vào DRIVE_DATA_DIR trước; USE_GOOGLE_DRIVE chỉ mount khi True.
Kaggle: Dataset nên chứa ảnh đã giải nén và labels/. Không ghi vào /kaggle/input.
Cell này cài đúng các gói bổ sung khi INSTALL_DEPS=True; không tải DeepWeeds hoặc weights.


In [ ]:
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "timm", "fvcore", "openpyxl", "pytest"], check=True)
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Mount Drive chỉ dành cho Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
if RUN_REPO_SETUP:
    setup_repo()

CODE_READY = (REPO_ROOT / "code" / "train.py").is_file()
if CODE_READY:
    for folder in (REPO_ROOT, REPO_ROOT / "code"):
        if str(folder) in sys.path:
            sys.path.remove(str(folder))
        sys.path.insert(0, str(folder))
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    import torchvision
    from torchvision import transforms
    from PIL import Image
    import dataset as data
    import model as models
    import losses
    import train
    import inference
    import benchmark
    import eval as evaluation
    for module in (data, models, losses, train, inference, benchmark):
        if Path(module.__file__).resolve().parent != (REPO_ROOT / "code").resolve():
            raise RuntimeError("Module cache trỏ sai repo; restart kernel rồi import lại.")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Python:", sys.version, "torch:", torch.__version__, "torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available(), "GPU:",
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
else:
    print("Chưa có repo. Bật RUN_REPO_SETUP rồi chạy lại phần 2–4.")

def require_code():
    if not CODE_READY:
        raise RuntimeError("Chuẩn bị repo/import ở phần 2–4 trước.")

def persistent_output():
    if PLATFORM_MODE == "colab" and not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Google Drive trước; không ghi run vào ổ /content tạm.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR


# 5. Kiểm tra MD5, giải nén và kiểm tra 17.509 ảnh
MD5 được tính theo chunk. Với Kaggle ảnh đã giải nén nhưng không có ZIP, ghi rõ MD5 ZIP chưa thể xác minh; vẫn kiểm tra đủ filename và nhãn bằng CSV chính thức.
ZIP được kiểm tra đường dẫn thành viên trước giải nén. Thư mục labels bắt buộc có labels.csv và đủ ba subset0.


In [ ]:
REQUIRED_LABELS = ["labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv"]
IMAGES_DIR = None
LABELS_DIR = None

def md5_chunks(path):
    digest = hashlib.md5()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_labels(root):
    matches = [p.parent for p in Path(root).rglob("labels.csv")
               if all((p.parent / name).is_file() for name in REQUIRED_LABELS)]
    if len(matches) != 1:
        raise RuntimeError(f"Cần đúng một thư mục labels có labels.csv và subset0: {matches}")
    return matches[0]

def find_images(root):
    jpgs = sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"})
    if len(jpgs) != EXPECTED_IMAGES:
        raise ValueError(f"Đếm được {len(jpgs)} ảnh; cần {EXPECTED_IMAGES}.")
    parents = {p.parent for p in jpgs}
    if len(parents) != 1:
        raise ValueError("Ảnh nằm nhiều thư mục; chọn Dataset có thư mục ảnh phẳng khớp Filename CSV.")
    return jpgs[0].parent, jpgs

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as zipped:
        for member in zipped.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        zipped.extractall(destination)

if RUN_PREPARE_DATA:
    require_code()
    if PLATFORM_MODE == "colab":
        source = Path(DRIVE_DATA_DIR)
        archive = source / "images.zip"
        if not archive.is_file() or not (source / "labels").is_dir():
            raise FileNotFoundError("Drive cần images.zip và labels/ trước khi copy.")
        for name in REQUIRED_LABELS:
            if not (source / "labels" / name).is_file():
                raise FileNotFoundError(name)
        if md5_chunks(archive) != EXPECTED_MD5:
            raise ValueError("MD5 images.zip trên Drive không khớp.")
        local = Path(LOCAL_DATA_DIR)
        local.mkdir(parents=True, exist_ok=True)
        copied_zip = local / "images.zip"
        if not copied_zip.exists() or md5_chunks(copied_zip) != EXPECTED_MD5:
            shutil.copy2(archive, copied_zip)
        if md5_chunks(copied_zip) != EXPECTED_MD5:
            raise ValueError("MD5 ZIP sau copy không khớp.")
        extraction = local / "extracted"
        extraction.mkdir(exist_ok=True)
        safe_extract(copied_zip, extraction)
        shutil.copytree(source / "labels", local / "labels", dirs_exist_ok=True)
        LABELS_DIR = find_labels(local / "labels")
        IMAGES_DIR, jpgs = find_images(extraction)
    else:
        source = Path(KAGGLE_DATA_DIR)
        if not source.is_dir():
            raise FileNotFoundError("Attach Dataset và điền KAGGLE_DATA_DIR.")
        LABELS_DIR = find_labels(source)
        archives = list(source.rglob("images.zip"))
        if archives:
            if len(archives) != 1 or md5_chunks(archives[0]) != EXPECTED_MD5:
                raise ValueError("ZIP không duy nhất hoặc MD5 không khớp.")
        else:
            print("Dataset đã giải nén; không có ZIP để xác minh MD5.")
        IMAGES_DIR, jpgs = find_images(source)
    persistent_output()
    print("IMAGES_DIR:", IMAGES_DIR, "LABELS_DIR:", LABELS_DIR, "images:", len(jpgs))
    print("Filename mẫu:", [p.name for p in jpgs[:5]])

def require_data():
    require_code()
    if IMAGES_DIR is None or LABELS_DIR is None:
        raise RuntimeError("Bật RUN_PREPARE_DATA và hoàn tất phần 5 trước.")


# 6. EDA và kiểm tra split
Chỉ dùng metadata split và ảnh train để minh họa. Biểu đồ nhãn có thể gồm ba split theo yêu cầu kiểm tra dữ liệu; không đọc prediction hoặc metric test.
Nhận xét sau khi chạy: **[Điền số lượng, tỷ lệ mất cân bằng và đặc điểm ảnh từ kết quả thật.]**


In [ ]:
def split_frames():
    require_data()
    frames = data.load_split(LABELS_DIR, fold=0)
    data.check_split(*frames, images_dir=IMAGES_DIR, expected_total=EXPECTED_IMAGES)
    return frames

if RUN_EDA:
    train_df, val_df, test_df = split_frames()
    counts = pd.DataFrame({name: frame["Label"].value_counts().reindex(range(9), fill_value=0)
                           for name, frame in zip(("train", "val", "test"), (train_df, val_df, test_df))})
    print("Số ảnh:", {name: len(frame) for name, frame in zip(counts.columns, (train_df, val_df, test_df))})
    display(counts)
    print("Tỷ lệ max/min:", counts.max() / counts.min().replace(0, np.nan))
    counts.plot.bar(title="DeepWeeds fold 0 — phân bố nhãn thật", ylabel="Số ảnh", xlabel="Label")
    plt.tight_layout()
    plt.show()
    fig, axes = plt.subplots(9, 3, figsize=(9, 24))
    for label in range(9):
        examples = train_df[train_df.Label == label].head(3)
        if len(examples) < 3:
            raise ValueError(f"Train lớp {label} thiếu ba ảnh minh họa.")
        for axis, filename in zip(axes[label], examples.Filename):
            with Image.open(IMAGES_DIR / filename) as image:
                axis.imshow(image.convert("RGB"))
            axis.set_title(f"{evaluation.CLASS_NAMES[label]}\n{filename}", fontsize=8)
            axis.axis("off")
    plt.tight_layout()
    plt.show()


# 7. Sanity check pipeline
Bật RUN_SANITY_CHECKS để xem batch train, ảnh sau augmentation và kiểm tra overfit một batch nhỏ.
CE ban đầu thường gần ln(9) ≈ 2.197 nhưng có thể lệch do phân bố logit/nhãn; đây là kỳ vọng chẩn đoán, không phải ngưỡng pass cứng.
Không ghi phép thử này vào bảng thí nghiệm chính thức.
Nhận xét sau khi chạy: **[Điền loss trước/sau và kiểm tra BatchNorm thực tế.]**


In [ ]:
if RUN_SANITY_CHECKS:
    require_data()
    train.set_seed(0)
    sanity_train, _, _ = split_frames()
    sanity_loader = data.make_loader(sanity_train, IMAGES_DIR, data.build_transforms(True, 224),
                                    batch_size=8, train=True, num_workers=0, pin_memory=False, seed=0)
    images, targets, names = next(iter(sanity_loader))
    print(images.shape, images.dtype, images.min().item(), images.max().item(), targets)
    mean = torch.tensor(data.IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(data.IMAGENET_STD).view(1, 3, 1, 1)
    display_images = (images * std + mean).clamp(0, 1)
    plt.figure(figsize=(12, 3))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i + 1)
        plt.imshow(display_images[i].permute(1, 2, 0))
        plt.title(f"Label {targets[i].item()}")
        plt.axis("off")
    plt.show()


In [ ]:
if RUN_SANITY_CHECKS:
    class SanityNet(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.bn = torch.nn.BatchNorm2d(3)
            self.head = torch.nn.Linear(3 * 32 * 32, 9)
        def get_classifier(self):
            return self.head
        def forward(self, x):
            return self.head(self.bn(x).flatten(1))

    small = torch.nn.functional.interpolate(images, size=(32, 32))
    sanity_net = SanityNet().cpu()
    criterion = losses.build_criterion("ce")
    optimizer = torch.optim.Adam(sanity_net.parameters(), lr=0.001)
    initial = criterion(sanity_net(small), targets).item()
    print("CE ban đầu:", initial, "tham chiếu ln(9):", float(np.log(9)))
    for step in range(80):
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(sanity_net(small), targets)
        loss.backward()
        optimizer.step()
    print("CE sau tối đa 80 bước:", criterion(sanity_net(small), targets).item())
    sanity_net.eval()
    assert not sanity_net.bn.training
    sanity_net.train()
    assert sanity_net.bn.training
    models.freeze_backbone(sanity_net)
    sanity_net.train()
    models.set_frozen_backbone_eval(sanity_net)
    assert not sanity_net.bn.training and all(p.requires_grad for p in sanity_net.head.parameters())
    del sanity_net, optimizer


# 8. Công thức baseline T00
Baseline được khai báo tường minh; RUN_BASELINE=False. Finetune cần tải pretrained weights khi bạn bật chạy thực sự.
Nếu OOM, giảm **cùng một** BASELINE_BATCH_SIZE cho tất cả backbone và ghi lý do vào báo cáo; không tự đổi batch giữa các run.
Artifact được ghi trực tiếp ra ổ bền. Run dở dang không được tự resume hoặc ghi đè.


In [ ]:
BASELINE = None
if CODE_READY:
    BASELINE = train.Config(
        exp_id="T00", seed=0, fold=0, backbone="resnet50", init="finetune",
        img_size=224, aug="basic", sampler=None, mix=None, loss="ce",
        epochs=12, batch_size=BASELINE_BATCH_SIZE, lr_backbone=1e-4, lr_head=1e-3,
        weight_decay=0.05, warmup_epochs=1, ema_decay=None, amp=True,
        save_test_predictions=False, expected_total=EXPECTED_IMAGES)

def config_for(exp_id, scope="experiments", **changes):
    require_data()
    root = persistent_output() / scope
    return replace(BASELINE, exp_id=exp_id, images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
                   out_dir=str(root / "runs"), pred_dir=str(root / "predictions"),
                   curve_dir=str(root / "curves"), **changes)

def write_json_new(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)

def run_record(cfg, category, difference):
    if cfg.save_test_predictions:
        raise ValueError("Mọi training run phải save_test_predictions=False.")
    record_path = train.run_dir(cfg) / "summary.json"
    if record_path.exists():
        record = json.loads(record_path.read_text(encoding="utf-8"))
        if record["config"] != asdict(cfg):
            raise ValueError("Config khác run đã lưu; chọn ID/output khác, không ghi đè.")
        for artifact in ("best.pt", "history.csv", "val_logits.npy"):
            if not (train.run_dir(cfg) / artifact).is_file():
                raise FileNotFoundError(artifact)
        print("Phục hồi summary:", record_path)
        return record
    if not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
        raise RuntimeError("Chưa có GPU. Chỉ bật ALLOW_CPU_TRAINING nếu chủ động chấp nhận chạy rất lâu.")
    try:
        summary = train.run(cfg)
        record = {"category": category, "difference": difference,
                  "config": asdict(cfg), "summary": summary}
        write_json_new(record_path, record)
        return record
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(f"OOM {cfg.exp_id}: batch={cfg.batch_size}. Dừng và điều chỉnh đồng loạt, không đổi tự động.")
        raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if RUN_BASELINE:
    run_record(config_for("T00"), "Training", "Baseline cố định theo GUIDE")


# 9. So sánh ít nhất 5 backbone
Các B01–B05 dùng cùng seed và toàn bộ công thức T00; chỉ đổi exp_id/backbone.
SELECTED_BACKBONE mặc định None. Điền sau khi xem validation cùng params/GMAC/thời gian.
Nhận xét sau khi chạy: **[Điền đối chiếu validation và chi phí; chưa có backbone thắng.]**


In [ ]:
BACKBONES = [
    ("B01", "resnet50"), ("B02", "resnext50_32x4d"), ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"), ("B05", "efficientnet_b0"),
]
if RUN_BACKBONES:
    for exp_id, backbone in BACKBONES:
        run_record(config_for(exp_id, backbone=backbone), "Backbones", "Chỉ đổi backbone so với T00")

def records(category=None):
    if not OUTPUT_DIR.exists():
        return []
    result = [json.loads(p.read_text(encoding="utf-8")) for p in OUTPUT_DIR.rglob("summary.json")]
    return [r for r in result if category is None or r.get("category") == category]

if CODE_READY:
    backbone_table = pd.DataFrame([dict(r["summary"], backbone=r["config"]["backbone"])
                                   for r in records("Backbones")])
    display(backbone_table if not backbone_table.empty else
            pd.DataFrame(columns=["exp_id", "backbone", "macro_f1_val", "top1_val", "params_m", "gmacs"]))


# 10. Các trục công thức huấn luyện
Recipe T00 dùng SELECTED_BACKBONE đã chọn; lưu trong namespace recipes để không đụng baseline ResNet50 ở phần 8.
A: frozen/scratch/finetune (finetune là T00). B: basic/color/randaug/Mixup/CutMix (basic là T00).
C: CE/label smoothing/focal/weighted CE (CE là T00). EMA là trục bổ sung.
Mỗi cấu hình đơn chỉ đổi một yếu tố khái niệm. T99 là kết hợp do người dùng điền COMBINED_RECIPE sau khi có số đo val.
Nhận xét sau khi chạy: **[Điền yếu tố thay đổi, bằng chứng cải thiện và thất bại.]**


In [ ]:
RECIPE_VARIANTS = [
    ("T01", "A: frozen", {"init": "frozen"}),
    ("T02", "A: scratch", {"init": "scratch"}),
    ("T03", "B: color", {"aug": "color"}),
    ("T04", "B: randaug", {"aug": "randaug"}),
    ("T05", "B: Mixup", {"mix": "mixup"}),
    ("T06", "B: CutMix", {"mix": "cutmix"}),
    ("T07", "C: label smoothing", {"loss": "ls", "label_smoothing": 0.1}),
    ("T08", "C: focal", {"loss": "focal"}),
    ("T09", "C: weighted CE", {"loss": "ce_weighted"}),
    ("T10", "D: EMA", {"ema_decay": 0.999}),
]
RECIPE_FIELDS = {"init", "aug", "sampler", "mix", "mix_alpha", "loss", "label_smoothing",
                 "focal_gamma", "class_weight_beta", "ema_decay", "drop_rate",
                 "lr_backbone", "lr_head", "weight_decay", "warmup_epochs", "epochs", "img_size"}

def validate_recipe(patch):
    if not isinstance(patch, dict) or set(patch) - RECIPE_FIELDS:
        raise ValueError("Recipe phải là dict chỉ chứa các field recipe được công bố.")

if RUN_TRAINING_ABLATIONS:
    if SELECTED_BACKBONE is None:
        raise ValueError("Điền SELECTED_BACKBONE sau khi xem validation backbone.")
    run_record(config_for("T00", scope="recipes", backbone=SELECTED_BACKBONE),
               "Training", "Recipe baseline: finetune/basic/CE")
    for exp_id, description, patch in RECIPE_VARIANTS:
        run_record(config_for(exp_id, scope="recipes", backbone=SELECTED_BACKBONE, **patch),
                   "Training", description)
    if COMBINED_RECIPE is not None:
        validate_recipe(COMBINED_RECIPE)
        run_record(config_for("T99", scope="recipes", backbone=SELECTED_BACKBONE, **COMBINED_RECIPE),
                   "Training", "Kết hợp các yếu tố đã chọn bằng validation")
    else:
        print("T99 chưa chọn; điền COMBINED_RECIPE từ kết quả val rồi chạy cell kết hợp riêng.")

# Có thể chạy riêng đoạn này khi T99 đã được chọn mà không lặp các run đơn.
def run_combined_recipe():
    if not RUN_TRAINING_ABLATIONS or SELECTED_BACKBONE is None or COMBINED_RECIPE is None:
        raise ValueError("Bật cờ và điền SELECTED_BACKBONE/COMBINED_RECIPE trước.")
    validate_recipe(COMBINED_RECIPE)
    return run_record(config_for("T99", scope="recipes", backbone=SELECTED_BACKBONE, **COMBINED_RECIPE),
                      "Training", "Kết hợp các yếu tố đã chọn bằng validation")


# 11. Các phương pháp suy luận — chỉ validation
Bật RUN_INFERENCE_EXPERIMENTS và điền INFERENCE_CHECKPOINT. Chỉ tạo loader validation.
I00: một view; I01: flip/prob; I02: five-crop; I03: flip/logit; I04: 256/288/320; I05: ensemble tùy chọn; I06: EMA nếu có; I07: temperature; I08: fusion.
ViT có thể từ chối resolution khác lúc train: lưu lỗi và nêu giới hạn, không đổi model để tạo số đo giả.
Khi gộp probability, lưu log(prob) làm logits tương đương để hiệu chuẩn một T trên phân phối đã gộp; raw logits vẫn dùng cho một view.
Nhận xét sau khi chạy: **[Điền macro-F1, top-1, ECE val và chi phí thực tế.]**


In [ ]:
INFERENCE_SPECS = {
    "I00": {"kind": "single"},
    "I01": {"kind": "hflip", "space": "prob"},
    "I02": {"kind": "fivecrop", "space": "prob"},
    "I03": {"kind": "hflip", "space": "logit"},
    "I04_256": {"kind": "single", "resolution": 256},
    "I04_288": {"kind": "single", "resolution": 288},
    "I04_320": {"kind": "single", "resolution": 320},
    "I05": {"kind": "ensemble"},
    "I06": {"kind": "single", "state": "ema"},
    "I07": {"kind": "single", "temperature": True},
    "I08": {"kind": "single", "fusion": True},
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_checkpoint(path, state="selected"):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint chưa có: {path}")
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    cfg = train.Config(**checkpoint["config"])
    key = checkpoint.get("selection_model", "raw") if state == "selected" else state
    weights = checkpoint.get("ema") if key == "ema" else checkpoint["model"]
    if weights is None:
        raise ValueError("Checkpoint không có EMA.")
    net = models.build_model(cfg.backbone, pretrained=False, num_classes=9,
                             drop_rate=cfg.drop_rate, init="scratch")
    net.load_state_dict(weights)
    return net.to(DEVICE).eval(), cfg

def method_model(path, spec, ensemble_paths=()):
    require_data()
    net, cfg = load_checkpoint(path, spec.get("state", "selected"))
    if spec.get("fusion", False):
        net = inference.fuse_conv_bn(net)
        print("Conv-BN pairs fused:", net._conv_bn_fusion_count)
    nets = [net]
    if spec["kind"] == "ensemble":
        if not ensemble_paths:
            raise ValueError("I05 cần ensemble checkpoints được chọn từ validation.")
        for other in ensemble_paths:
            other_net, other_cfg = load_checkpoint(other)
            if other_cfg.img_size != cfg.img_size:
                raise ValueError("Ensemble này yêu cầu cùng img_size; chọn checkpoint tương thích.")
            nets.append(other_net)
    size = spec.get("resolution", cfg.img_size)
    kind = spec["kind"]
    class ViewsModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.nets = torch.nn.ModuleList(nets)
        def forward(self, x):
            if kind == "single":
                return self.nets[0](x)
            if kind == "ensemble":
                logits = [m(x) for m in self.nets]
            else:
                views = ([x, inference.view_hflip(x)] if kind == "hflip"
                         else inference.views_multicrop(x, size))
                logits = [self.nets[0](v) for v in views]
            if spec.get("space", "prob") == "logit":
                return torch.stack(logits).mean(0)
            probabilities = inference.aggregate_views(logits, space="prob")
            return torch.from_numpy(np.log(np.clip(probabilities, 1e-300, 1))).float()
    wrapper = ViewsModel().eval()
    if kind == "fivecrop":
        transform = transforms.Compose([
            transforms.Resize((size + 32, size + 32)), transforms.ToTensor(),
            transforms.Normalize(data.IMAGENET_MEAN, data.IMAGENET_STD)])
    elif "resolution" in spec:
        transform = transforms.Compose([
            transforms.Resize((size, size)), transforms.ToTensor(),
            transforms.Normalize(data.IMAGENET_MEAN, data.IMAGENET_STD)])
    else:
        transform = data.build_transforms(False, size)
    k = len(nets) if kind == "ensemble" else 5 if kind == "fivecrop" else 2 if kind == "hflip" else 1
    return wrapper, transform, cfg, k

def method_loader(frame, transform):
    return data.make_loader(frame, IMAGES_DIR, transform, INFERENCE_BATCH_SIZE,
                            train=False, shuffle=False, num_workers=0, pin_memory=DEVICE.type == "cuda")

def evaluate_validation_method(path, spec, destination, method_id, ensemble_paths=()):
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(f"Artifact inference đã tồn tại: {destination}")
    net, transform, cfg, k = method_model(path, spec, ensemble_paths)
    validation = data.load_split(LABELS_DIR, fold=0)[1]
    names, labels, logits = inference.predict_logits(net, method_loader(validation, transform), DEVICE)
    temperature = inference.fit_temperature(logits, labels) if spec.get("temperature", False) else 1.0
    probs = inference.apply_temperature(logits, temperature)
    metrics = evaluation.compute_metrics(labels, probs.argmax(1), probs)
    destination.mkdir(parents=True, exist_ok=False)
    np.save(destination / "val_logits.npy", logits)
    np.save(destination / "val_probabilities.npy", probs)
    evaluation.save_predictions(destination / f"{cfg.exp_id}_seed{cfg.seed}_val.csv", names, labels, probs)
    record = {"method": method_id, "K": k, "spec": spec, "temperature": temperature,
              "checkpoint": str(Path(path).resolve()), "checkpoint_sha256": file_sha256(path),
              "ensemble_paths": [str(Path(p).resolve()) for p in ensemble_paths],
              "ensemble_sha256": [file_sha256(p) for p in ensemble_paths],
              "exp_id": cfg.exp_id, "seed": cfg.seed, "backbone": cfg.backbone,
              "macro_f1_val": metrics["macro_f1"], "top1_val": metrics["top1"], "ece_val": metrics["ece"]}
    write_json_new(destination / "inference_record.json", record)
    del net
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record

if RUN_INFERENCE_EXPERIMENTS:
    require_data()
    if INFERENCE_CHECKPOINT is None:
        raise ValueError("Điền INFERENCE_CHECKPOINT từ checkpoint đã chọn bằng val.")
    for method_id, spec in INFERENCE_SPECS.items():
        if method_id == "I05" and not ENSEMBLE_CHECKPOINTS:
            print("Bỏ qua I05: chưa chọn ensemble.")
            continue
        destination = persistent_output() / "inference" / method_id
        try:
            evaluate_validation_method(INFERENCE_CHECKPOINT, spec, destination, method_id,
                                       ENSEMBLE_CHECKPOINTS if method_id == "I05" else ())
        except (ValueError, RuntimeError) as exc:
            print(method_id, "không chạy được:", exc)
            write_json_new(persistent_output() / "inference_errors" /
                           f"{method_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json",
                           {"method": method_id, "error": str(exc)})
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    rows = [json.loads(p.read_text(encoding="utf-8"))
            for p in (OUTPUT_DIR / "inference").glob("*/inference_record.json")]
    display(pd.DataFrame(rows, columns=["method", "K", "macro_f1_val", "top1_val", "ece_val"]))


# 12. Đo độ trễ
RUN_LATENCY=False. Báo cáo CUDA dùng warmup ≥10, iterations ≥50; CPU chỉ là **smoke, không phải latency triển khai chính thức**.
Đo batch 1 và 32, FP32/AMP, FP16 chỉ trên CUDA. OOM batch32 được ghi là lỗi, không đổi batch.
TTA benchmark chỉ đo K forward cùng độ phân giải; không tính preprocessing hoặc aggregation, không dùng số này như latency đầy đủ của multicrop/multiscale.
Nhận xét sau khi chạy: **[Điền thiết bị, p50/p95/p99, thông lượng và điều kiện đo thật.]**


In [ ]:
if RUN_LATENCY:
    require_data()
    if INFERENCE_CHECKPOINT is None:
        raise ValueError("Chọn checkpoint trước khi đo latency.")
    if LATENCY_WARMUP < 10 or LATENCY_ITERATIONS < 50:
        raise ValueError("Báo cáo yêu cầu warmup>=10 và iterations>=50.")
    net, cfg = load_checkpoint(INFERENCE_CHECKPOINT)
    modes = ["fp32", "amp"] + (["fp16"] if DEVICE.type == "cuda" else [])
    for batch in (1, 32):
        for dtype in modes:
            tag = f"batch{batch}_{dtype}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
            try:
                report = benchmark.latency_report(net, batch, cfg.img_size, dtype=dtype,
                    device=DEVICE, warmup=LATENCY_WARMUP, iters=LATENCY_ITERATIONS)
                report.update({"checkpoint": str(INFERENCE_CHECKPOINT),
                               "official": DEVICE.type == "cuda", "method": "single"})
                write_json_new(persistent_output() / "latency" / f"{tag}.json", report)
                print(report)
            except RuntimeError as exc:
                print(tag, "không đo được:", exc)
                write_json_new(persistent_output() / "latency_errors" / f"{tag}.json", {"error": str(exc)})
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
    if LATENCY_K_VIEWS is not None:
        report = benchmark.tta_latency(net, LATENCY_K_VIEWS, batch_size=1, img_size=cfg.img_size,
             device=DEVICE, warmup=LATENCY_WARMUP, iters=LATENCY_ITERATIONS)
        report.update({"official": DEVICE.type == "cuda", "method": "K_forward_only",
                       "checkpoint": str(INFERENCE_CHECKPOINT)})
        write_json_new(persistent_output() / "latency" /
                       f"tta_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.json", report)
    del net


# 13. Chọn cấu hình cuối hoàn toàn bằng validation
Điền FINAL_BACKBONE, FINAL_TRAINING_CONFIG và FINAL_INFERENCE_METHOD tại cell cấu hình; không đặt sẵn người thắng.
Không xem test để chọn. Khi chênh lệch nhỏ hơn std, không khẳng định cấu hình tốt hơn.
Cell khóa in cấu hình và ghi immutable manifest trước final training. Đổi cấu hình sau đó bị từ chối.
I05 final cần thêm FINAL_ENSEMBLE_CHECKPOINTS theo seed, được chọn trước test.
Nhận xét sau khi chạy: **[Điền bằng chứng validation và lý do khóa cấu hình.]**


In [ ]:
def final_selection():
    if FINAL_BACKBONE is None or FINAL_TRAINING_CONFIG is None or FINAL_INFERENCE_METHOD is None:
        raise ValueError("Điền FINAL_BACKBONE, FINAL_TRAINING_CONFIG, FINAL_INFERENCE_METHOD bằng validation.")
    validate_recipe(FINAL_TRAINING_CONFIG)
    if FINAL_INFERENCE_METHOD not in INFERENCE_SPECS:
        raise ValueError("FINAL_INFERENCE_METHOD phải là ID trong INFERENCE_SPECS.")
    if INFERENCE_SPECS[FINAL_INFERENCE_METHOD].get("state") == "ema" and FINAL_TRAINING_CONFIG.get("ema_decay") is None:
        raise ValueError("I06 cần chọn recipe có ema_decay trước khi final training.")
    if len(FINAL_SEEDS) < 3 or len(set(FINAL_SEEDS)) != len(FINAL_SEEDS):
        raise ValueError("Cần ít nhất ba seed khác nhau cho cả final và baseline.")
    if FINAL_INFERENCE_METHOD == "I05":
        for seed in FINAL_SEEDS:
            if not FINAL_ENSEMBLE_CHECKPOINTS.get(seed):
                raise ValueError("I05 final cần checkpoint ensemble cho mỗi seed.")
    return {"backbone": FINAL_BACKBONE, "recipe": FINAL_TRAINING_CONFIG,
            "method": FINAL_INFERENCE_METHOD, "spec": INFERENCE_SPECS[FINAL_INFERENCE_METHOD],
            "seeds": FINAL_SEEDS, "baseline": asdict(BASELINE),
            "ensemble": {str(k): v for k, v in FINAL_ENSEMBLE_CHECKPOINTS.items()}}

def ensure_lock(create=False):
    require_data()
    selected = final_selection()
    path = persistent_output() / "final" / "locked_config.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != selected:
            raise ValueError("Cấu hình hiện tại khác manifest đã khóa; không được dùng test để đổi cấu hình.")
    elif create:
        write_json_new(path, selected)
    else:
        raise FileNotFoundError("Khóa cấu hình và chạy final training trước.")
    print(json.dumps(selected, indent=2, ensure_ascii=False))
    return selected

if CODE_READY:
    validation_table = pd.DataFrame([
        dict(r["summary"], category=r["category"], backbone=r["config"]["backbone"],
             scope=Path(r["config"]["out_dir"]).parent.name)
        for r in records() if r["category"] != "Final"])
    display(validation_table if not validation_table.empty else
            pd.DataFrame(columns=["exp_id", "seed", "macro_f1_val", "top1_val"]))

if RUN_FINAL_TRAINING or RUN_FINAL_TEST:
    ensure_lock(create=RUN_FINAL_TRAINING)


# 14. Chạy nhiều seed
Final F01 và mốc T00 cùng FINAL_SEEDS=[0,1,2], checkpoint chọn bằng validation.
save_test_predictions luôn False. Sau mỗi run lưu summary và validation của phương pháp đã khóa.
Fit temperature (nếu chọn) tại đây, lưu T theo seed để test dùng lại. Chưa mở test ở phần này.
Phục hồi phiên: đọc summary/manifest từ output; run dở dang báo lỗi, không tự ghi đè.


In [ ]:
def final_jobs():
    selected = final_selection()
    jobs = []
    for seed in FINAL_SEEDS:
        jobs.append((config_for("T00", scope="final", seed=seed), "I00", INFERENCE_SPECS["I00"], []))
        cfg = config_for("F01", scope="final", seed=seed, backbone=FINAL_BACKBONE, **FINAL_TRAINING_CONFIG)
        extras = FINAL_ENSEMBLE_CHECKPOINTS.get(seed, []) if FINAL_INFERENCE_METHOD == "I05" else []
        jobs.append((cfg, FINAL_INFERENCE_METHOD, selected["spec"], extras))
    return jobs

def selected_val_dir(cfg):
    return OUTPUT_DIR / "final" / "selected_validation" / f"{cfg.exp_id}_seed{cfg.seed}"

if RUN_FINAL_TRAINING:
    ensure_lock(create=True)
    for cfg, method_id, spec, extras in final_jobs():
        run_record(cfg, "Final", "Validation-only checkpoint selection; final configuration locked")
        checkpoint = train.run_dir(cfg) / "best.pt"
        destination = selected_val_dir(cfg)
        record_path = destination / "inference_record.json"
        if record_path.exists():
            existing = json.loads(record_path.read_text(encoding="utf-8"))
            if (existing["spec"] != spec or existing["checkpoint_sha256"] != file_sha256(checkpoint)
                    or existing["ensemble_sha256"] != [file_sha256(p) for p in extras]):
                raise ValueError("Selected validation artifact không khớp checkpoint/spec.")
        else:
            evaluate_validation_method(checkpoint, spec, destination, method_id, extras)
    selected_rows = [json.loads((selected_val_dir(cfg) / "inference_record.json").read_text(encoding="utf-8"))
                     for cfg, _, _, _ in final_jobs()]
    table = pd.DataFrame(selected_rows)
    display(table[["exp_id", "seed", "macro_f1_val", "top1_val", "ece_val"]])
    display(table.groupby("exp_id")[["macro_f1_val", "top1_val", "ece_val"]].agg(["mean", "std"]))


# 15. Final test được khóa xác nhận
Chỉ bật sau khi đã đọc manifest và kiểm tra artifact validation của cả ba seed:
RUN_FINAL_TEST=True, CONFIRM_FINAL_TEST="I_HAVE_LOCKED_THE_CONFIGURATION", CONFIRM_NO_TEST_PREDICTIONS=True.
Xác nhận thứ ba nghĩa là người dùng đã kiểm tra và xác nhận chưa có prediction test.
Preflight từ chối nếu CSV, logits hoặc marker test đã có. Marker được tạo **trước** forward và được giữ nếu lỗi/ngắt phiên để ngăn tự chạy lại test.
Mỗi model/seed chỉ duyệt test loader một lần; TTA/ensemble có K forward trên mỗi batch đúng phương pháp đã khóa.
Nếu lỗi giữa chừng, giữ artifact và ghi báo cáo sự cố; không tự xóa marker rồi thử cấu hình khác.
Không có metric test nào tham gia chọn checkpoint, nhiệt độ hay phương pháp.


In [ ]:
if RUN_FINAL_TEST:
    if CONFIRM_FINAL_TEST != "I_HAVE_LOCKED_THE_CONFIGURATION" or not CONFIRM_NO_TEST_PREDICTIONS:
        raise PermissionError("Cần chuỗi xác nhận khóa final và xác nhận chưa có test predictions.")
    locked = ensure_lock(create=False)
    jobs = final_jobs()
    plans = []
    for cfg, method_id, spec, extras in jobs:
        checkpoint = train.run_dir(cfg) / "best.pt"
        validation_path = selected_val_dir(cfg) / "inference_record.json"
        if not checkpoint.is_file() or not validation_path.is_file():
            raise FileNotFoundError("Thiếu checkpoint/validation calibration cho toàn bộ final seeds.")
        validation = json.loads(validation_path.read_text(encoding="utf-8"))
        if validation["spec"] != spec or validation["checkpoint_sha256"] != file_sha256(checkpoint):
            raise ValueError("Checkpoint hoặc phương pháp đã đổi sau validation.")
        if validation["ensemble_sha256"] != [file_sha256(p) for p in extras]:
            raise ValueError("Ensemble checkpoint đã đổi sau validation.")
        for extra in extras:
            extra_meta = torch.load(extra, map_location="cpu", weights_only=True)
            if extra_meta["config"]["seed"] != cfg.seed:
                raise ValueError("Ensemble final cần checkpoint đúng seed.")
        marker = OUTPUT_DIR / "final" / "test_attempts" / f"{cfg.exp_id}_seed{cfg.seed}.json"
        tensor_dir = OUTPUT_DIR / "final" / "test_arrays" / f"{cfg.exp_id}_seed{cfg.seed}"
        prediction = train.pred_path(cfg, "test")
        uncalibrated = Path(cfg.pred_dir) / f"{cfg.exp_id}uncal_seed{cfg.seed}_test.csv"
        if any(p.exists() for p in (marker, tensor_dir, prediction, uncalibrated)):
            raise FileExistsError(f"Test đã có artifact/marker: {cfg.exp_id} seed {cfg.seed}; dừng.")
        plans.append((cfg, spec, extras, checkpoint, validation, marker, tensor_dir, prediction, uncalibrated))

    # Chỉ sau khi toàn bộ preflight đạt mới đọc test và tạo test loader.
    test_frame = data.load_split(LABELS_DIR, fold=0)[2]
    for cfg, spec, extras, checkpoint, validation, marker, tensor_dir, prediction, uncalibrated in plans:
        net, transform, _, _ = method_model(checkpoint, spec, extras)
        write_json_new(marker, {"status": "started", "checkpoint": str(checkpoint),
                               "locked_config": locked, "started_utc": datetime.now(timezone.utc).isoformat()})
        test_loader = method_loader(test_frame, transform)
        names, labels, logits = inference.predict_logits(net, test_loader, DEVICE)
        tensor_dir.mkdir(parents=True, exist_ok=False)
        np.save(tensor_dir / "test_logits.npy", logits)
        before = inference.apply_temperature(logits, 1.0)
        after = inference.apply_temperature(logits, validation["temperature"])
        np.save(tensor_dir / "test_probabilities.npy", after)
        evaluation.save_predictions(prediction, names, labels, after)
        evaluation.save_predictions(uncalibrated, names, labels, before)
        write_json_new(tensor_dir / "completed.json",
                       {"prediction": str(prediction), "temperature_from_val": validation["temperature"]})
        del net, test_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# 16. eval.py score/grade
Các lệnh được tạo bằng subprocess, chỉ chạy khi RUN_EVAL_REPORTS=True và final test đã hoàn tất.
Tương đương: `python eval.py score ...` và `python eval.py grade ...`.
Điền EVAL_UNCAL_PATTERN cho --uncal, EVAL_FINAL_VAL_PATTERN cho --final-val; LATENCY_P95_MS cho --latency-p95-ms chỉ từ phép đo CUDA batch1 đúng cấu hình.
Không có p95 thật thì bỏ option đó. Không sửa eval.py.


In [ ]:
if RUN_EVAL_REPORTS:
    ensure_lock(create=False)
    final_root = OUTPUT_DIR / "final"
    for cfg, _, _, _ in final_jobs():
        prediction = train.pred_path(cfg, "test")
        completed = final_root / "test_arrays" / f"{cfg.exp_id}_seed{cfg.seed}" / "completed.json"
        if not prediction.is_file() or not completed.is_file():
            raise FileNotFoundError("Chỉ score/grade sau khi final test hoàn tất mọi seed.")
    common = ["--test-csv", str(LABELS_DIR / "test_subset0.csv"),
              "--labels", str(LABELS_DIR / "labels.csv")]
    for exp_id in ("F01", "T00"):
        subprocess.run([sys.executable, str(REPO_ROOT / "eval.py"), "score",
                        "--pred", str(final_root / "predictions" / f"{exp_id}_seed*_test.csv"),
                        *common, "--tag", exp_id, "--out", str(final_root / "eval_out")], check=True)
    command = [sys.executable, str(REPO_ROOT / "eval.py"), "grade",
               "--final", str(final_root / "predictions" / "F01_seed*_test.csv"),
               "--baseline", str(final_root / "predictions" / "T00_seed*_test.csv"),
               *common, "--out", str(final_root / "eval_out")]
    import glob
    for option, pattern in (("--uncal", EVAL_UNCAL_PATTERN), ("--final-val", EVAL_FINAL_VAL_PATTERN)):
        if pattern is not None:
            if not glob.glob(str(pattern)):
                raise FileNotFoundError(f"{option}: không tìm thấy prediction.")
            command.extend([option, str(pattern)])
    if EVAL_FINAL_VAL_PATTERN is not None:
        command.extend(["--val-csv", str(LABELS_DIR / "val_subset0.csv")])
    if LATENCY_P95_MS is not None:
        if not torch.cuda.is_available():
            raise ValueError("Không dùng số CPU smoke làm latency triển khai chính thức.")
        command.extend(["--latency-p95-ms", str(LATENCY_P95_MS), "--latency-method", "proper"])
    subprocess.run(command, check=True)


# 17. Tạo results.xlsx
Bảy sheet lấy dữ liệu từ summary, validation inference, latency JSON và output eval.py thật. Khi chưa có artifact, sheet chỉ có header.
Summary chỉ xếp hạng **validation**, không dùng test. Mean/std qua seed có ddof=1; một seed để std trống.
Workbook freeze header, auto-filter và format số. Nếu results.xlsx có sẵn, tạo file có timestamp trừ khi đã xác nhận ghi đè.


In [ ]:
SHEET_COLUMNS = {
    "Backbones": ["exp_id", "backbone", "seed", "macro_f1_val", "top1_val", "params_m", "gmacs",
                  "mean_epoch_seconds", "difference", "source"],
    "Training": ["exp_id", "backbone", "seed", "difference", "macro_f1_val", "top1_val", "source"],
    "Inference": ["exp_id", "seed", "method", "K", "macro_f1_val", "top1_val", "ece_val", "source"],
    "Final": ["exp_id", "seed", "top1", "macro_f1", "ece", "n", "source"],
    "PerClass": ["exp_id", "class", "precision_mean", "precision_std", "recall_mean", "recall_std",
                 "f1_mean", "f1_std", "support", "source"],
    "Latency": ["method", "device", "gpu", "dtype", "batch", "img_size", "p50_ms", "p95_ms", "p99_ms",
                "mean_ms", "images_per_s", "n", "includes_preprocessing", "official", "source"],
    "Summary": ["category", "scope", "backbone", "exp_id", "method", "n_seeds", "macro_f1_val_mean", "macro_f1_val_std",
                "top1_val_mean", "top1_val_std"],
}

def workbook_frames():
    rows = {name: [] for name in SHEET_COLUMNS}
    validation_rows = []
    for path in OUTPUT_DIR.rglob("summary.json"):
        record = json.loads(path.read_text(encoding="utf-8"))
        category = record["category"]
        row = dict(record["summary"], backbone=record["config"]["backbone"],
                   scope=Path(record["config"]["out_dir"]).parent.name,
                   difference=record["difference"], source=str(path))
        if category in ("Backbones", "Training"):
            rows[category].append(row)
            validation_rows.append(dict(row, category=category, method="training_checkpoint"))
    for path in OUTPUT_DIR.rglob("inference_record.json"):
        row = dict(json.loads(path.read_text(encoding="utf-8")), source=str(path))
        rows["Inference"].append(row)
        validation_rows.append(dict(row, category="Inference",
                                    scope="final_selected" if "selected_validation" in path.parts else "inference"))
    for path in (OUTPUT_DIR / "latency").glob("*.json"):
        row = json.loads(path.read_text(encoding="utf-8"))
        row.update({key + "_ms": row[key] for key in ("p50", "p95", "p99", "mean")})
        rows["Latency"].append(dict(row, source=str(path)))
    eval_root = OUTPUT_DIR / "final" / "eval_out"
    for exp_id in ("F01", "T00"):
        seed_path = eval_root / f"{exp_id}_per_seed.csv"
        class_path = eval_root / f"{exp_id}_per_class.csv"
        if seed_path.is_file():
            rows["Final"].extend(dict(row, exp_id=exp_id, source=str(seed_path))
                                 for row in pd.read_csv(seed_path).to_dict("records"))
        if class_path.is_file():
            rows["PerClass"].extend(dict(row, exp_id=exp_id, source=str(class_path))
                                    for row in pd.read_csv(class_path).to_dict("records"))
    if validation_rows:
        table = pd.DataFrame(validation_rows)
        for (category, scope, backbone, exp_id, method), group in table.groupby(
                ["category", "scope", "backbone", "exp_id", "method"]):
            if group["seed"].duplicated().any():
                raise ValueError("Không gộp nhiều cấu hình của cùng seed thành mean/std.")
            rows["Summary"].append({
                "category": category, "scope": scope, "backbone": backbone, "exp_id": exp_id, "method": method,
                "n_seeds": group["seed"].nunique(),
                "macro_f1_val_mean": group["macro_f1_val"].mean(),
                "macro_f1_val_std": group["macro_f1_val"].std(ddof=1),
                "top1_val_mean": group["top1_val"].mean(), "top1_val_std": group["top1_val"].std(ddof=1)})
    frames = {name: pd.DataFrame(rows[name]).reindex(columns=columns) for name, columns in SHEET_COLUMNS.items()}
    frames["Summary"] = frames["Summary"].sort_values("macro_f1_val_mean", ascending=False)
    return frames

if RUN_EXPORT_RESULTS:
    require_code()
    root = persistent_output()
    destination = root / "results.xlsx"
    if destination.exists() and not CONFIRM_OVERWRITE_RESULTS:
        destination = root / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.xlsx"
    with pd.ExcelWriter(destination, engine="openpyxl") as writer:
        for sheet, frame in workbook_frames().items():
            frame.to_excel(writer, sheet_name=sheet, index=False)
            worksheet = writer.sheets[sheet]
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            for column in worksheet.columns:
                worksheet.column_dimensions[column[0].column_letter].width = min(45, max(15, len(str(column[0].value)) + 2))
                for cell in column[1:]:
                    if isinstance(cell.value, float):
                        cell.number_format = "0.0000"
    print("Workbook:", destination)


# 18. Checklist artifact và hướng dẫn lưu kết quả
- Kiểm tra config.json, history.csv, best.pt, summary.json, curve PNG và validation predictions sau mỗi run.
- Final cần locked_config.json, calibration validation theo seed, test_attempts, completed.json và F01/T00 predictions cho cùng ba seed.
- Copy/tải results.xlsx, curves/, predictions/, report.md hoặc report.pdf và notebook chạy thật từ OUTPUT_DIR. Curves/predictions nằm theo namespace experiments/recipes/final; gom vào thư mục bài nộp và giữ ID/seed rõ ràng.
- Kaggle: Save Version để giữ /kaggle/working output; tải Output trước khi kết thúc phiên nếu cần. Colab: output đã ghi thẳng Drive.
- Khôi phục phiên: mount/attach dữ liệu, import lại, bật chuẩn bị dữ liệu, dùng đúng output và cấu hình cũ. Run hoàn tất phục hồi từ summary; run dở dang cần xem log và chọn run mới có ghi chú.
- Không commit images.zip, dataset hoặc checkpoint lớn. Notebook không tự push GitHub.
- Notebook tracked: execution_count=None, outputs=[]. Save a copy in Drive cho bản có kết quả thật.
Nhận xét sau khi chạy: **[Điền artifact còn thiếu, giới hạn số seed/GPU và các thí nghiệm thất bại.]**


In [ ]:
if RUN_ARTIFACT_CHECK:
    if not OUTPUT_DIR.exists():
        print("Chưa có artifact.")
    else:
        for path in sorted(OUTPUT_DIR.rglob("*")):
            if path.is_file():
                print(path.relative_to(OUTPUT_DIR), path.stat().st_size, "bytes")
